In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os
os.chdir("/home/j/jackbarker/surface_predictor")
import pickle

import scripts.analyse_eofs as analyse_eofs
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Checking some other variables

In [ ]:
def plot_errors_vs_24h_diff(variable_name, level=None, start_idx=0, n_steps=5, error_vmin=None, error_vmax=None, diff_vmin=None, diff_vmax=None):
 
    """
    Plot GraphCast errors vs 24h differences for any variable
    Only plots 6am timesteps
    
    Parameters:
    variable_name: Name of the variable to plot (e.g., '2m_temperature', 'geopotential', 'total_precipitation_6hr')
    start_idx: Starting index for the time series (default 0)
    n_steps: Number of timesteps to show (default 5)
    error_vmin, error_vmax: Color scale limits for errors
    diff_vmin, diff_vmax: Color scale limits for 24h differences
    """
    
    # Load the datasets for the requested variable
    def preprocess(ds):
        if "batch" in ds.dims:
            ds = ds.squeeze(dim="batch")
        return ds

    # Load forecast data
    forecast_dataset = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,).sel(level=level)[variable_name].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    forecast_dataset = forecast_dataset.rename({"time": "datetime"})
    forecast_dataset = forecast_dataset.set_index(datetime="datetime")
    
    # Load error data
    error_dataset = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,).sel(level=level)[variable_name].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    error_dataset = error_dataset.set_index(datetime="datetime")
    
    # Calculate ERA5/truth data
    era5_dataset = forecast_dataset - error_dataset
    
    # Filter for 6am times only
    era5_6am = era5_dataset.sel(datetime=era5_dataset.datetime.dt.hour == 6)
    errors_6am = error_dataset.sel(datetime=error_dataset.datetime.dt.hour == 6)
    gc_6am = forecast_dataset.sel(datetime=forecast_dataset.datetime.dt.hour == 6)
    temp_errors_6am = eth_temp_errors.sel(datetime=eth_temp_errors.datetime.dt.hour == 6)
    
    # Create figure with 4 rows
    fig, axs = plt.subplots(4, n_steps, figsize=(4*n_steps, 12), 
                           subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Get the time slice from 6am data only
    end_idx = start_idx + n_steps
    era5_slice = era5_6am.isel(datetime=slice(start_idx, end_idx))
    gc_slice = gc_6am.isel(datetime=slice(start_idx, end_idx))
    era5_prev_slice = era5_6am.isel(datetime=slice(start_idx-1, end_idx-1))
    gc_prev_slice = gc_6am.isel(datetime=slice(start_idx-1, end_idx-1))
    error_slice = errors_6am.isel(datetime=slice(start_idx, end_idx))
    temp_error_slice = temp_errors_6am.isel(datetime=slice(start_idx, end_idx))

    # Align datetime indices
    era5_prev_slice = era5_prev_slice.assign_coords(datetime=era5_slice.datetime)
    gc_prev_slice = gc_prev_slice.assign_coords(datetime=gc_slice.datetime)
    
    # Calculate 24h differences
    era5_24h_diff = era5_slice - era5_prev_slice
    gc_24h_diff = gc_slice - gc_prev_slice

    # Auto-calculate color scales if not provided
    if error_vmin is None or error_vmax is None:
        error_abs_max = float(np.max(np.abs(error_slice.values)))
        error_vmin = -error_abs_max
        error_vmax = error_abs_max
        
    if diff_vmin is None or diff_vmax is None:
        # Calculate max absolute value from both ERA5 and GC differences
        era5_diff_abs_max = float(np.max(np.abs(era5_24h_diff.values)))
        gc_diff_abs_max = float(np.max(np.abs(gc_24h_diff.values)))
        diff_abs_max = max(era5_diff_abs_max, gc_diff_abs_max)
        diff_vmin = -diff_abs_max
        diff_vmax = diff_abs_max
    
    # Temp errors at very top

    for i in range(n_steps):
        ax = axs[0, i]
        im = ax.imshow(temp_error_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-5, vmax=5)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"Temperature Error: {str(temp_error_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for errors
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('Temperature Error (K)', rotation=270, labelpad=15)
    
    # Plot GraphCast errors on top row
    for i in range(n_steps):
        ax = axs[1, i]
        im = ax.imshow(error_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=error_vmin, vmax=error_vmax)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"{variable_name} Error: {str(error_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for errors
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label(f'{variable_name} Error', rotation=270, labelpad=15)

    # Plot ERA5 24h differences on middle row
    for i in range(n_steps):
        ax = axs[2, i]
        im = ax.imshow(era5_24h_diff.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=diff_vmin, vmax=diff_vmax)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"ERA5 24h diff: {str(era5_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for 24h differences
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label(f'24h {variable_name} Change', rotation=270, labelpad=15)

    # Plot GraphCast 24h differences on bottom row
    for i in range(n_steps):
        ax = axs[3, i]
        im = ax.imshow(gc_24h_diff.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=diff_vmin, vmax=diff_vmax)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"GC 24h diff: {str(gc_slice.datetime.data[i])[:13]}")

        # Add colorbar for 24h differences
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label(f'24h {variable_name} Change', rotation=270, labelpad=15)
    
    plt.suptitle(f"GraphCast Errors vs 24h {variable_name} Changes (6am only)", 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_errors_vs_24h_diff('specific_humidity', level=1000, start_idx=1, n_steps=8, error_vmin=-0.002, error_vmax=0.002,)


In [ ]:
plot_errors_vs_24h_diff('10m_u_component_of_wind', level=1000, start_idx=1, n_steps=8, error_vmin=-2, error_vmax=2,)


In [ ]:
plot_errors_vs_24h_diff('10m_v_component_of_wind', level=1000, start_idx=1, n_steps=8, error_vmin=-2, error_vmax=2,)


In [ ]:
plot_errors_vs_24h_diff('total_precipitation_6hr', level=1000, start_idx=1, n_steps=8, error_vmin=-0.003, error_vmax=0.003, diff_vmin=-0.05, diff_vmax=0.05)


# Seeing if wind forecasts make a difference

In [ ]:
def plot_temp_errors_vs_wind_u(start_idx=0, n_steps=5):
    """
    Plot 2m temperature errors on top row and 10m u-component wind ERA5 values on bottom row
    Only plots 6am timesteps
    
    Parameters:
    start_idx: Starting index for the time series (default 0)
    n_steps: Number of timesteps to show (default 5)
    """
    
    # Load the datasets
    def preprocess(ds):
        if "batch" in ds.dims:
            ds = ds.squeeze(dim="batch")
        return ds

    # Load 10m u-component wind forecast data
    wind_u_forecast = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,)["10m_u_component_of_wind"].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    wind_u_forecast = wind_u_forecast.rename({"time": "datetime"})
    wind_u_forecast = wind_u_forecast.set_index(datetime="datetime")
    
    # Load 10m u-component wind error data
    wind_u_errors = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,)["10m_u_component_of_wind"].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    wind_u_errors = wind_u_errors.set_index(datetime="datetime")
    
    # Calculate ERA5 u-component wind (truth)
    wind_u_era5 = wind_u_forecast - wind_u_errors
    
    # Filter for 6am times only
    temp_errors_6am = eth_temp_errors.sel(datetime=eth_temp_errors.datetime.dt.hour == 6)
    wind_u_era5_6am = wind_u_era5.sel(datetime=wind_u_era5.datetime.dt.hour == 6)
    
    # Create figure with 2 rows
    fig, axs = plt.subplots(2, n_steps, figsize=(4*n_steps, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Get the time slice from 6am data only
    end_idx = start_idx + n_steps
    temp_error_slice = temp_errors_6am.isel(datetime=slice(start_idx, end_idx))
    wind_u_era5_slice = wind_u_era5_6am.isel(datetime=slice(start_idx, end_idx))
    
    # Auto-calculate color scales
    temp_error_abs_max = float(np.max(np.abs(temp_error_slice.values)))
    wind_u_abs_max = float(np.max(np.abs(wind_u_era5_slice.values)))
    
    # Plot 2m temperature errors on top row
    for i in range(n_steps):
        ax = axs[0, i]
        im = ax.imshow(temp_error_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-temp_error_abs_max, vmax=temp_error_abs_max)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"2m Temp Error: {str(temp_error_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for temperature errors
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('Temperature Error (K)', rotation=270, labelpad=15)
    
    # Plot 10m u-component wind ERA5 values on bottom row
    for i in range(n_steps):
        ax = axs[1, i]
        im = ax.imshow(wind_u_era5_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-wind_u_abs_max, vmax=wind_u_abs_max)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"10m U-wind ERA5: {str(wind_u_era5_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for wind
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('U-component Wind Speed (m/s)', rotation=270, labelpad=15)
    
    plt.suptitle(f"2m Temperature Errors vs 10m U-component Wind ERA5 (6am only)", 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Print the auto-calculated scales for reference
    print(f"Auto-calculated scales:")
    print(f"Temperature error range: [{-temp_error_abs_max:.3f}, {temp_error_abs_max:.3f}] K")
    print(f"U-component wind range: [{-wind_u_abs_max:.3f}, {wind_u_abs_max:.3f}] m/s")

# Example usage
plot_temp_errors_vs_wind_u(start_idx=1, n_steps=8)

In [ ]:
def plot_temp_errors_vs_wind_u_all_times(start_idx=0, n_steps=5):
    """
    Plot 2m temperature errors on top row and 10m u-component wind ERA5 values on bottom row
    Plots all times (not filtered to 6am only)
    
    Parameters:
    start_idx: Starting index for the time series (default 0)
    n_steps: Number of timesteps to show (default 5)
    """
    
    # Load the datasets
    def preprocess(ds):
        if "batch" in ds.dims:
            ds = ds.squeeze(dim="batch")
        return ds

    # Load 10m u-component wind forecast data
    wind_u_forecast = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,)["10m_u_component_of_wind"].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    wind_u_forecast = wind_u_forecast.rename({"time": "datetime"})
    wind_u_forecast = wind_u_forecast.set_index(datetime="datetime")
    
    # Load 10m u-component wind error data
    wind_u_errors = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,)["10m_u_component_of_wind"].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    wind_u_errors = wind_u_errors.set_index(datetime="datetime")
    
    # Calculate ERA5 u-component wind (truth)
    wind_u_era5 = wind_u_forecast - wind_u_errors
    
    # Create figure with 2 rows (no 6am filtering)
    fig, axs = plt.subplots(2, n_steps, figsize=(4*n_steps, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Get the time slice (all times, not just 6am)
    end_idx = start_idx + n_steps
    temp_error_slice = eth_temp_errors.isel(datetime=slice(start_idx, end_idx))
    wind_u_era5_slice = wind_u_era5.isel(datetime=slice(start_idx, end_idx))
    
    # Auto-calculate color scales
    temp_error_abs_max = float(np.max(np.abs(temp_error_slice.values)))
    wind_u_abs_max = float(np.max(np.abs(wind_u_era5_slice.values)))
    
    # Plot 2m temperature errors on top row
    for i in range(n_steps):
        ax = axs[0, i]
        im = ax.imshow(temp_error_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-temp_error_abs_max, vmax=temp_error_abs_max)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"2m Temp Error: {str(temp_error_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for temperature errors
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('Temperature Error (K)', rotation=270, labelpad=15)
    
    # Plot 10m u-component wind ERA5 values on bottom row
    for i in range(n_steps):
        ax = axs[1, i]
        im = ax.imshow(wind_u_era5_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-wind_u_abs_max, vmax=wind_u_abs_max)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"10m U-wind ERA5: {str(wind_u_era5_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for wind
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('U-component Wind Speed (m/s)', rotation=270, labelpad=15)
    
    plt.suptitle(f"2m Temperature Errors vs 10m U-component Wind ERA5 (all times)", 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()
    
    # Print the auto-calculated scales for reference
    print(f"Auto-calculated scales:")
    print(f"Temperature error range: [{-temp_error_abs_max:.3f}, {temp_error_abs_max:.3f}] K")
    print(f"U-component wind range: [{-wind_u_abs_max:.3f}, {wind_u_abs_max:.3f}] m/s")

# Example usage - now shows all times, not just 6am
plot_temp_errors_vs_wind_u_all_times(start_idx=8, n_steps=10)

# Cumulative precipitation

In [ ]:
def plot_temp_errors_vs_24h_precip(start_idx=0, n_steps=5, roll_period=4):
    """
    Plot 2m temperature errors on top row and 24hr cumulative precipitation on bottom row
    24hr precipitation is calculated by summing the previous 4 6hr precipitation values
    
    Parameters:
    start_idx: Starting index for the time series (default 0)
    n_steps: Number of timesteps to show (default 5)
    """
    
    # Load the datasets
    def preprocess(ds):
        if "batch" in ds.dims:
            ds = ds.squeeze(dim="batch")
        return ds

    # Load 6hr precipitation forecast data
    precip_forecast = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,)["total_precipitation_6hr"].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    precip_forecast = precip_forecast.rename({"time": "datetime"})
    precip_forecast = precip_forecast.set_index(datetime="datetime")
    
    # Load 6hr precipitation error data
    precip_errors = xr.open_mfdataset(
        [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_{t.strftime('%Y%m%d-%H')}_n0.nc" for t in analyse_eofs.get_times(2016, 2016, 1, 1, 6, 0, 15, 25)], 
        concat_dim="datetime", 
        combine="nested", 
        preprocess=preprocess,)["total_precipitation_6hr"].sel(lat=slice(-35,38), lon=slice(0,52)).load()
    
    precip_errors = precip_errors.set_index(datetime="datetime")
    
    # Calculate ERA5 precipitation (truth)
    precip_era5 = precip_forecast - precip_errors
    
    # Calculate 24hr cumulative precipitation by summing previous 4 timesteps
    # Use rolling window with window=4 and sum
    precip_24h = precip_era5.rolling(datetime=roll_period, min_periods=roll_period).sum()

    precip_24h_6am = precip_24h.sel(datetime=precip_24h.datetime.dt.hour == 6)
    eth_temp_errors_6am = eth_temp_errors.sel(datetime=eth_temp_errors.datetime.dt.hour == 6)
    
    # Create figure with 2 rows
    fig, axs = plt.subplots(2, n_steps, figsize=(4*n_steps, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Get the time slice (start from index 3 to ensure we have 4 previous timesteps for 24h sum)
    start_with_history = max(start_idx, roll_period-1)
    end_idx = start_with_history + n_steps
    temp_error_slice = eth_temp_errors_6am.isel(datetime=slice(start_idx, end_idx))
    precip_24h_slice = precip_24h_6am.isel(datetime=slice(start_idx, end_idx))
    
    # Auto-calculate color scales
    temp_error_abs_max = float(np.max(np.abs(temp_error_slice.values)))
    
    # Plot 2m temperature errors on top row
    for i in range(n_steps):
        ax = axs[0, i]
        im = ax.imshow(temp_error_slice.isel(datetime=i), 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-temp_error_abs_max, vmax=temp_error_abs_max)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"2m Temp Error: {str(temp_error_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for temperature errors
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('Temperature Error (K)', rotation=270, labelpad=15)
    
    # Plot 24hr cumulative precipitation on bottom row
    for i in range(n_steps):
        ax = axs[1, i]
        im = ax.imshow(precip_24h_slice.isel(datetime=i), 
                      cmap="Blues", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=0, vmax=0.025)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"{roll_period*6}h Precip: {str(precip_24h_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for precipitation
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label(f'{roll_period*6}h Precipitation (m)', rotation=270, labelpad=15)
    
    plt.suptitle(f"2m Temperature Errors vs {roll_period*6}hr Cumulative Precipitation", 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()

    return precip_24h
    
    # Print the auto-calculated scales for reference

# Example usage
precip_24hr = plot_temp_errors_vs_24h_precip(start_idx=0, n_steps=10, roll_period=8)

# Soil moisture

In [ ]:
import xarray_regrid
ds = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/soil_moisture_data/era5_data/src_and_evap.nc")
# rename valid_time to datetime
ds = ds.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})
ds = ds.rename({"evavt": "veg_evap", "e": "tot_evap"})
# ds = ds.sel(lat=(ds.lat>=-35), lon=slice(0,52))
# ds = ds.sel(lat=(ds.lat<38))
ds = ds.reindex(lat = list(reversed(ds.lat)))
ds = ds.drop_vars(["expver", "number"])
ds = ds.sel(lat=slice(-35, 38), lon=slice(0, 52))

src_data = ds["src"]
veg_evap_data = ds["veg_evap"]
tot_evap_data = ds["tot_evap"]

In [ ]:
def plot(start_idx=0, n_steps=5, comp_dataset=None, data_name="", data_hour=6, vmin=0, vmax=1, cmap="Blues"):

    
    # Filter for 6am times only
    era5_data = comp_dataset.sel(datetime=comp_dataset.datetime.dt.hour == data_hour)
    era5_data = era5_data.sel(datetime=era5_data.datetime.dt.day >= 15)
    era5_jan_mean = era5_data.groupby('datetime.month').mean(dim='datetime').sel(month=1)
    era5_data = era5_data-era5_jan_mean

    errors_6am = eth_temp_errors.sel(datetime=eth_temp_errors.datetime.dt.hour == 6)
    
    
    # Create figure with 2 rows
    
    
    # Get the time slice from 6am data only
    end_idx = start_idx + n_steps
    era5_slice = era5_data.isel(datetime=slice(start_idx, end_idx))
    error_slice = errors_6am.isel(datetime=slice(start_idx, end_idx))
    

    # Plot SRC data on top row
    fig, axs = plt.subplots(2, n_steps, figsize=(4*n_steps, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})
    for i in range(n_steps):
        ax = axs[0,i]
        im = ax.imshow(era5_slice.isel(datetime=i), 
                      cmap=cmap, 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=vmin, vmax=vmax)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"{data_name} anomaly at {data_hour}h: {str(era5_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for deviations
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)


    # fig, axs = plt.subplots(1, n_steps, figsize=(4*n_steps, 4), 
    #                        subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Plot errors on bottom row  
    for i in range(n_steps):
        ax = axs[1,i]
        im = ax.imshow(error_slice.isel(datetime=i).to_dataarray()[0], 
                      cmap="RdBu_r", 
                      transform=ccrs.PlateCarree(), 
                      extent=[0, 52, -35, 38], 
                      origin="lower",
                      vmin=-5, vmax=5)
        
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.set_title(f"GraphCast Error at 6hr: {str(error_slice.datetime.data[i])[:13]}")
        
        # Add colorbar for errors
        if i == n_steps - 1:
            cbar = plt.colorbar(im, ax=ax, shrink=0.8)
            cbar.set_label('Temperature Error (K)', rotation=270, labelpad=15)
    
    plt.suptitle(f"Errors at 6hr and {data_name} anomaly at {data_hour}hr", 
                 fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()

# Example usage
plot(start_idx=0, n_steps=11, comp_dataset=veg_evap_data, data_name="Veg Evap", data_hour=6, vmin=-0.0001, vmax=0.0001, cmap="RdBu_r")

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=tot_evap_data, data_name="Tot Evap", data_hour=6, vmin=-0.0004, vmax=0.0004, cmap="RdBu_r")

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=tot_evap_data, data_name="Tot Evap", data_hour=0, vmin=-0.0004, vmax=0.0004, cmap="RdBu_r")

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=src_data, data_name="SRC", data_hour=6, vmin=-0.0004, vmax=0.0004, cmap="RdBu_r")

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=src_data, data_name="SRC", data_hour=0, vmin=-0.0004, vmax=0.0004, cmap="RdBu_r")

## Clouds

In [ ]:
cloud_cover = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/soil_moisture_data/era5_data/data/data_stream-oper_stepType-instant.nc")
cloud_cover = cloud_cover.drop_vars(["number", "expver"])
cloud_cover

cloud_cover = cloud_cover.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})

cloud_cover = cloud_cover.reindex(lat = list(reversed(cloud_cover.lat)))
cloud_cover = cloud_cover.sel(lat=slice(-35, 38), lon=slice(0, 52))

cloud_cover = cloud_cover["tcc"]

In [ ]:
cloud_cover.isel(datetime=0).plot()

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=cloud_cover, data_name="Cloud Cover", data_hour=6, vmin=-1, vmax=1, cmap="RdBu_r")

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=cloud_cover, data_name="Cloud Cover", data_hour=0, vmin=-1, vmax=1, cmap="RdBu_r")

In [ ]:
evaporation = xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/soil_moisture_data/era5_data/data/data_stream-oper_stepType-accum.nc")
evaporation = evaporation.drop_vars(["number", "expver"])


evaporation = evaporation.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})
evaporation = evaporation.reindex(lat = list(reversed(evaporation.lat)))
evaporation = evaporation.sel(lat=slice(-35, 38), lon=slice(0, 52))

evaporation = evaporation["e"]

In [ ]:
evaporation.isel(datetime=0).plot()

In [ ]:
plot(start_idx=0, n_steps=11, comp_dataset=evaporation, data_name="Evaporation", data_hour=6, vmin=-0.0001, vmax=0.0001, cmap="RdBu_r")